# Customer Intelligence Engine

## 02. Data Cleaning and Preparation

### Objective

This notebook focuses on cleaning and preparing the
Online Retail II dataset for business analysis.

Based on the initial data exploration, several data
quality issues were identified, including:

- Duplicate records
- Cancelled transactions
- Negative quantities
- Zero or negative prices
- Missing Customer IDs

The goal is to prepare separate datasets for
sales analysis and customer analytics while
keeping the original data unchanged.

## 1. Import Libraries

Import the libraries needed for data analysis.

In [1]:
import pandas as pd
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load the original dataset

Load both worksheets from the original Online Retail II dataset and combine them into one DataFrame.

In [3]:
# Define the file location
file_path = Path("/Users/richardnguyenk/Downloads/GitHub Project/customer-intelligence-engine/data/raw/online_retail_II.xlsx")

# Load both worksheets
retail_data = pd.read_excel(
    file_path,
    sheet_name=None
)

# Combine both years
df = pd.concat(
    retail_data.values(),
    ignore_index=True
)

# Check dataset size
print("Original dataset:", df.shape)

Original dataset: (1067371, 8)


## 3. Remove duplicate records

The initial analysis identified 34,335 duplicate rows.

For this project, I will remove exact duplicate rows to reduce the risk of double-counting transactions.

This assumes identical rows represent duplicated records, although the dataset does not contain a unique transaction-line identifier to confirm this.


In [4]:
# Remove exact duplicates row
df_clean = df.drop_duplicates().copy()

# Compare dataset sizes after removing duplicates
print("Original rows:", len(df))
print("Rows after removing duplicates:", len(df_clean))
print("Number of duplicates removed:", len(df) - len(df_clean))

Original rows: 1067371
Rows after removing duplicates: 1033036
Number of duplicates removed: 34335


## 4. Identify Cancelled Transactions

Some invoices begin with the letter C, indicating cancelled or credit-related transactions.

These records should be seperated from completed sales to avoid mixing sales with returns or reversals.



In [5]:
# Identify cancelled invoices
cancelled = df_clean[
    df_clean["Invoice"].astype(str).str.startswith("C")
].copy()

# Check number of cancelled records
print("Cancelled transaction rows:", len(cancelled))

Cancelled transaction rows: 19104


## 5. Prepare Completed Sales

For the sales analysis, transactions that meet the following conditions will be kept:

- Invoice does not begin with C
- Quantity is greater than zero
- Price is greater than zero

These conditions provide a consistent definition of completed sales for this project

Other transactions will be kept separately for further investigation.

In [6]:
# Keep completed sales transactions
sales = df_clean[
    (~df_clean["Invoice"].astype(str).str.startswith("C")) &
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] > 0)
].copy()

print("Completed sales rows:", len(sales))

Completed sales rows: 1007913


## 6. Calculate Sales Revenue

Create a new variable called TotalSales

TotalSales represents the value of each transaction line and is calculated as:

TotalSales = Quantity x Price

This variable will be used to analyse sales performance, product revenue and customer spending.



In [7]:
# Calculate transaction-line revenue
sales["TotalSales"] = sales["Quantity"] * sales["Price"]

# Display the results
sales[["Invoice", "Quantity", "Price", "TotalSales"]].head()

,Invoice,Quantity,Price,TotalSales
0,489434,12,6.95,83.4
1,489434,12,6.75,81.0
2,489434,12,6.75,81.0
3,489434,48,2.10,100.8
4,489434,24,1.25,30.0


## 7. Prepare Customer Data

Customer-level analysis requires a valid Customer ID.

A separate dataset containing only completed sales with avaialble Customer IDs will be created.

Records without Customer IDs will remain in the sales dataset.



In [8]:
# Keep sales with available Customer IDs
customers = sales.dropna(subset=["Customer ID"]).copy()

# Compare dataset sizes
print("Sales records:", len(sales))
print("Customer records:", len(customers))

Sales records: 1007913
Customer records: 779425


In [9]:
# Convert Customer IDs to text
customers["Customer ID"] = customers["Customer ID"].astype(int).astype(str)

# Check results
customers[["Customer ID", "Invoice", "TotalSales"]].head()

,Customer ID,Invoice,TotalSales
0,13085,489434,83.4
1,13085,489434,81.0
2,13085,489434,81.0
3,13085,489434,100.8
4,13085,489434,30.0


## 8. Validate Cleaned Data

Check that the cleaned datasets meet the
requirements for sales and customer analysis.

Will confirm that:
- Duplicate records have been removed
- Sales quantities are positive
- Sales prices are positive
- Customer IDs are available in the customer dataset

In [10]:
# Check remaining duplicates
print("Duplicate rows:", sales.duplicated().sum())

# Check invalid quantities
print("Invalid quantities:", (sales["Quantity"] <= 0).sum())

# Check invalid prices
print("Invalid prices:", (sales["Price"] <= 0).sum())

# Check missing customer IDs
print("Missing Customer IDs:", customers["Customer ID"].isnull().sum())

Duplicate rows: 0
Invalid quantities: 0
Invalid prices: 0
Missing Customer IDs: 0


## 9. Export Cleaned Datasets

Save the cleaned datasets as CSV files.

These files will be used in the next stages
of the Customer Intelligence Engine project.

In [11]:
# Create processed folder if needed
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Save completed sales
sales.to_csv(
    "data/processed/sales_clean.csv",
    index=False
)

# Save customer transactions
customers.to_csv(
    "data/processed/customers_clean.csv",
    index=False
)

# Save cancellation records for future analysis
cancelled.to_csv(
    "data/processed/cancelled_transactions.csv",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.



## 10. Conclusion

The data cleaning process reduced the original dataset from 1,067,371 records to 1,007,913 completed-sales records, retaining approximately 94.4% of the original data.

After excluding transactions without Customer IDs, 779,425 records remained available for customer-level analysis.

The cleaned sales dataset retains transactions without Customer IDs to support overall sales and product analysis. The customer dataset contains only identifiable customers and will be used for customer segmentation and purchasing behaviour analysis.

The original dataset remains unchanged, and cancellation records have been saved separately for further investigation.

### Next Step

Notebook 03 will focus on Exploratory Data Analysis (EDA), including sales performance, monthly revenue trends, best-selling products and geographical sales patterns.